In [58]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import os
import multiscale_phate as mp
import scprep

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import scprep

h5ad_path = os.path.expanduser("/Users/raghu/yale/data/B_Cells_with_flowjo_populations_FINAL.h5ad")
print(h5ad_path)

In [ ]:
# This h5ad file holds RAW per-cell flow cytometry data for the B cell panel: ~69.7M cells x 85
# detector channels, pooled across 171 FCS files ("samples"), with FlowJo gating calls for every
# cell stored in obs (e.g. "flowjo_population_clean"). Channel names are raw FCS detector names
# (e.g. "Comp-YL1-A"), not marker names -- the $PnS stain-name mapping wasn't kept when this was
# converted from FCS.
#
# X alone is ~24GB as float32, so we can't anndata.read_h5ad() this directly (it'll get OOM-killed).
# Instead we read it with h5py and take a random subsample of cells per sample, since each sample
# occupies one contiguous block of rows in X -- that keeps every read contiguous and keeps peak
# memory to one sample's block (~400MB) at a time.

n_cells_per_sample = 150   # ~150 cells x 171 samples ≈ 25k cells, same scale as the CSV panels
random_state = 1

In [ ]:
rng = np.random.default_rng(random_state)

with h5py.File(h5ad_path, "r") as f:
    channel_names = [c.decode() if isinstance(c, bytes) else c for c in f["var/_index"][:]]

    sample_codes = f["obs/sample/codes"][:]
    sample_categories = [c.decode() if isinstance(c, bytes) else c for c in f["obs/sample/categories"][:]]

    pop_codes = f["obs/flowjo_population_clean/codes"][:]
    pop_categories = [c.decode() if isinstance(c, bytes) else c for c in f["obs/flowjo_population_clean/categories"][:]]

    # Each sample occupies one contiguous block of rows -> find the block boundaries.
    change_points = np.flatnonzero(np.diff(sample_codes)) + 1
    block_starts = np.concatenate(([0], change_points))
    block_ends = np.concatenate((change_points, [len(sample_codes)]))

    rows, sample_labels, population_labels = [], [], []

    for start, end in zip(block_starts, block_ends):
        block_size = end - start
        n_pick = min(n_cells_per_sample, block_size)
        picked = np.sort(rng.choice(block_size, size=n_pick, replace=False))

        rows.append(f["X"][start:end][picked])
        sample_labels.append(np.full(n_pick, sample_categories[sample_codes[start]]))
        population_labels.append(np.array(pop_categories)[pop_codes[start:end][picked]])

X_sub = np.concatenate(rows, axis=0)
sample_labels = np.concatenate(sample_labels)
population_labels = np.concatenate(population_labels)

flow_data = pd.DataFrame(X_sub, columns=channel_names)
flow_data["sample"] = sample_labels
flow_data["flowjo_population_clean"] = population_labels

print(flow_data.shape)

In [62]:
flow_data

,clinical_id,CD3,CD4,FSC,Fixable Aqua,GranzymeB,IL-6,IL17a,IL4,SSC,TNFa,scatter
CD183/CXCR3,,,,,,,,,,,,
87.201891,INP.1.0135,101.813858,64.225411,75.042587,82.770324,66.556649,54.398252,67.890583,65.743299,29.644287,72.241951,39.503285
68.831717,INP.1.0025,101.179621,108.937434,65.660113,80.369513,66.352506,70.147092,74.055899,59.705775,33.801739,57.230513,51.479867
74.704822,INP.1.0017,102.605253,104.306948,63.633076,83.601536,68.816906,60.221645,74.350488,62.797595,28.146556,67.395183,44.232587
73.030574,INP.1.0115,110.917774,58.958663,56.297488,73.842694,66.046914,61.781231,67.197866,63.816282,29.188888,69.931605,51.847585
73.341908,INP.1.0135,73.341908,66.021766,74.471277,96.013300,71.549688,64.169653,80.687593,76.072993,42.197704,67.549285,56.663060
...,...,...,...,...,...,...,...,...,...,...,...,...
73.234745,INP.1.0148,111.036541,112.426838,58.319248,78.826464,68.025770,61.325934,68.265721,63.936705,31.060344,79.343365,53.259163
66.971206,INP.1.0164,110.525550,95.226796,60.442968,88.847345,71.676543,59.627970,74.356960,61.909273,46.923312,69.363785,77.632375
72.858202,INP.1.0074,105.673496,107.194056,63.861895,78.524290,65.611860,55.597877,66.103341,64.742790,37.952163,99.684059,59.428495


In [ ]:
# Use scatter (FSC-A/SSC-A) plus the compensated fluorescence channels (Comp-*-A); skip the
# raw/uncompensated channels and the -H/-W pulse-shape variants, which are redundant for clustering.
feature_cols = ["FSC-A", "SSC-A"] + [c for c in channel_names if c.startswith("Comp-") and c.endswith("-A")]
flow_data_numeric = flow_data[feature_cols]
flow_data_numeric

In [ ]:
# Raw flow data is on a huge linear scale (negative compensated values up to ~1e6), which would
# dominate PCA/diffusion potential over the more informative low-magnitude channels. Apply the
# standard arcsinh transform used for flow/CyTOF data before running Multiscale PHATE.
flow_data_transformed = scprep.transform.arcsinh(flow_data_numeric, cofactor=150)
flow_data_transformed

In [ ]:
mp_op = mp.Multiscale_PHATE(random_state=1, n_jobs=-1)
mp_op.fit_transform(flow_data_transformed)

In [65]:
levels = mp_op.levels
print(levels)
visualization_level = 1
cluster_level = 8

[0, 12, 14, 22, 62, 73, 76, 79, 89, 102, 115, 131, 137, 141, 156]


In [66]:
coarse_embedding, coarse_clusters, coarse_sizes = mp_op.transform(visualization_level = levels[visualization_level],
                                                                  cluster_level = levels[cluster_level])

In [ ]:
scprep.plot.scatter2d(coarse_embedding, 
                      s = 2*np.sqrt(coarse_sizes), 
                      c = coarse_clusters, 
                      legend=True,
                      ticks=False,
                      label_prefix="msPHATE", 
                      fontsize=14,
                      figsize=(15,15),
                      title='B Cells (FlowJo)')
plt.show()

In [ ]:
# Get the expression vector for your channel of interest.
# Channels here are raw detector names, not marker names, but the FlowJo gating categories hint at
# a few mappings (see flow_data["flowjo_population_clean"].unique()), e.g. YL1-A ~ CD180, YL3-A ~ CD86.
marker = "Comp-YL1-A"
marker_vector = flow_data_transformed[marker].values

# Map channel expression to the coarse embedding
coarse_expression = mp_op.get_expression(
    marker_vector,
    visualization_level=levels[visualization_level]
)

In [ ]:
scprep.plot.scatter2d(
    coarse_embedding,
    c=coarse_expression,
    s=2*np.sqrt(coarse_sizes),
    cmap="viridis",
    colorbar=True,
    label_prefix="msPHATE",
    fontsize=14,
    figsize=(15,15),
    title=f"{marker} expression"
)
plt.show()